# Lesson 1 Notebook

This is the **R version** of `Lesson01_A` (Network course). It is designed to run in **Jupyter with an R kernel (IRkernel)**.

> If you opened this notebook and the kernel is not available, run the installation steps in the next section (once) and then restart Jupyter.


## (One-time) Install & register the R kernel (IRkernel)

Run this cell **only if** you don't already have an R kernel in Jupyter.

In [ ]:
# One-time IRkernel install (uncomment if needed)
# install.packages("IRkernel")
# IRkernel::installspec(user = TRUE)  # registers the kernel for your user


## Introduction to NetworkX  ➜  igraph/tidygraph in R

In Python you used **NetworkX**. In R we'll mostly use:

- **igraph** for core graph data structures & algorithms
- **tidygraph** + **ggraph** for tidy workflows and plotting
- **tidyverse** for data wrangling


## The `import` statement  ➜  `library(...)` in R

### Exercise
1. Load the required libraries.
2. If a package is missing, install it.


In [2]:
# If needed (run once):
# install.packages(c("igraph","tidyverse","tidygraph","ggraph"))

library(igraph)
library(tidyverse)
library(tidygraph)
library(ggraph)


NameError: name 'library' is not defined

## Creating and drawing simple graphs

### Goal
Create an undirected graph, inspect basic properties, and plot it.

### Exercise 1
1. Create a graph with nodes A–F and a few edges.
2. Compute number of nodes/edges, density, and degree.
3. Plot it.


In [ ]:
# 1) Create a simple undirected graph
edges <- tribble(
  ~from, ~to,
  "A",   "B",
  "A",   "C",
  "B",   "C",
  "C",   "D",
  "D",   "E",
  "E",   "F"
)

g <- graph_from_data_frame(edges, directed = FALSE)

# 2) Basic properties
vcount(g)                 # number of nodes
ecount(g)                 # number of edges
edge_density(g)           # density
degree(g)                 # degree per node

# 3) Plot (ggraph)
set.seed(1)
as_tbl_graph(g) |>
  ggraph(layout = "fr") +
  geom_edge_link(alpha = 0.5) +
  geom_node_point(size = 3) +
  geom_node_text(aes(label = name), repel = TRUE, size = 3) +
  theme_void()


## Creating non-simple Graphs - Edge properties

In Python you explored:
- Directed graphs
- Weighted graphs
- Custom edge attributes & MultiGraphs

In R/igraph we can represent all of these. igraph also allows **multiple edges** between the same nodes (a multigraph).


### Directed graphs

### Exercise 2
1. Create a directed graph.
2. Compute in-degree and out-degree.
3. Check weak/strong connectivity.


In [ ]:
edgesD <- tribble(
  ~from, ~to,
  "A",   "B",
  "A",   "C",
  "C",   "A",
  "C",   "D",
  "D",   "E"
)

gD <- graph_from_data_frame(edgesD, directed = TRUE)

degree(gD, mode = "in")
degree(gD, mode = "out")

is_connected(gD, mode = "weak")
is_connected(gD, mode = "strong")


### Weighted graphs

### Exercise 3
1. Create a weighted graph from a data frame.
2. Compute node **strength** (weighted degree).
3. Plot with edge width mapped to weight.


In [ ]:
edgesW <- tribble(
  ~from, ~to, ~weight,
  "A",   "B",  3,
  "A",   "C",  1,
  "B",   "C",  2,
  "C",   "D",  5
)

gW <- graph_from_data_frame(edgesW, directed = FALSE)
E(gW)$weight <- edgesW$weight

strength(gW, weights = E(gW)$weight)

set.seed(1)
as_tbl_graph(gW) |>
  ggraph(layout = "fr") +
  geom_edge_link(aes(width = weight), alpha = 0.5) +
  geom_node_point(size = 3) +
  geom_node_text(aes(label = name), repel = TRUE, size = 3) +
  theme_void()


### Custom attributes & MultiGraphs

### Exercise 4
1. Create a graph with a **custom edge attribute** (e.g., interaction type).
2. Create a **multigraph** (multiple edges between same pair).
3. Explore `is_simple()` and simplify the graph while preserving weights.


In [ ]:
edgesA <- tribble(
  ~from, ~to, ~interaction, ~weight,
  "A",   "B",  "friend",      1,
  "A",   "B",  "colleague",   2,   # multiple edge A-B -> multigraph
  "B",   "C",  "friend",      1,
  "C",   "D",  "family",      3
)

gM <- graph_from_data_frame(edgesA, directed = FALSE)
E(gM)$interaction <- edgesA$interaction
E(gM)$weight <- edgesA$weight

is_simple(gM)  # FALSE means parallel edges or loops exist

# Simplify by merging multiple edges (sum weights)
g_simple <- simplify(gM, remove.multiple = TRUE, remove.loops = TRUE,
                     edge.attr.comb = list(weight = "sum", interaction = "concat"))

E(g_simple)$weight
E(g_simple)$interaction


## Creating non-simple Graphs - Node properties

### Exercise 5
1. Add node attributes (e.g., group, color label, body size).
2. Use node attributes in computations and plots.


In [ ]:
gN <- g

# Add attributes
V(gN)$group <- c("G1","G1","G1","G2","G2","G2")
V(gN)$size_trait <- c(2.1, 1.8, 3.0, 2.5, 1.2, 2.9)

# Inspect
vertex_attr_names(gN)
V(gN)$group
V(gN)$size_trait

# Plot using attributes
set.seed(1)
as_tbl_graph(gN) |>
  activate(nodes) |>
  mutate(group = as.factor(group)) |>
  ggraph(layout = "fr") +
  geom_edge_link(alpha = 0.4) +
  geom_node_point(aes(shape = group), size = 3) +
  geom_node_text(aes(label = name), repel = TRUE, size = 3) +
  theme_void()


### Bipartite graphs

In Python/NetworkX you handled bipartite sets and projections.
In igraph, bipartite graphs are represented by a logical vertex attribute `type`.

We'll build one from an incidence matrix.


#### Obtaining the bipartite sets

### Exercise 6
1. Create an incidence matrix (plants × pollinators).
2. Build a bipartite graph.
3. Verify the two sets.


In [ ]:
# Incidence matrix (rows = plants, cols = pollinators)
M <- matrix(c(
  1,0,1,
  0,1,1,
  1,1,0
), nrow = 3, byrow = TRUE)

rownames(M) <- c("Plant1","Plant2","Plant3")
colnames(M) <- c("Poll1","Poll2","Poll3")

gB <- graph_from_incidence_matrix(M, weighted = TRUE)

# Vertex sets: type == FALSE corresponds to rows, TRUE to columns
table(V(gB)$type)

plants <- V(gB)[V(gB)$type == FALSE]$name
polls  <- V(gB)[V(gB)$type == TRUE]$name

plants
polls


#### Visualizing bipartite networks

### Exercise 7
Plot the bipartite network with a bipartite layout.


In [ ]:
set.seed(1)
as_tbl_graph(gB) |>
  ggraph(layout = "bipartite") +
  geom_edge_link(alpha = 0.6) +
  geom_node_point(aes(shape = as.factor(type)), size = 3) +
  geom_node_text(aes(label = name), repel = TRUE, size = 3) +
  theme_void()


#### One mode projection of bipartite networks

### Exercise 8
1. Project the bipartite network into plant-plant and pollinator-pollinator networks.
2. Compute degree distributions of each projection.


In [ ]:
proj <- bipartite_projection(gB)
g_plants <- proj$proj1
g_polls  <- proj$proj2

degree(g_plants) |> summary()
degree(g_polls)  |> summary()


# Working with network files & formats

This section mirrors reading/writing graphs and converting between graphs and data frames.


### Save to file

### Exercise 9
1. Save a graph to GraphML.
2. Read it back.


In [ ]:
# Choose a path you can write to
out_graphml <- "lesson01_example.graphml"

write_graph(g, out_graphml, format = "graphml")
g2 <- read_graph(out_graphml, format = "graphml")

g2


### Networks as Dataframes

### Exercise 10
1. Convert edges and nodes to data frames.
2. Rebuild the graph from those tables.


In [ ]:
edges_df <- as_data_frame(g, what = "edges") |> as_tibble()
nodes_df <- as_data_frame(g, what = "vertices") |> as_tibble()

edges_df
nodes_df

g_rebuilt <- graph_from_data_frame(edges_df, directed = FALSE, vertices = nodes_df)
isomorphic(g, g_rebuilt)


### Reading from files

We'll cover two common cases:
- Reading a matrix (adjacency or incidence)
- Reading an interaction list (edge list)


#### Reading a Matrix

### Exercise 11
1. Create an adjacency matrix.
2. Build a graph from it.


In [ ]:
A <- matrix(c(
  0,1,1,0,
  1,0,1,0,
  1,1,0,1,
  0,0,1,0
), nrow = 4, byrow = TRUE)

rownames(A) <- colnames(A) <- paste0("N", 1:4)

gA <- graph_from_adjacency_matrix(A, mode = "undirected", diag = FALSE)
gA


#### Reading an interaction list

### Exercise 12
1. Save an edge list CSV.
2. Read it and build a graph.


In [ ]:
# Create and save an edge list
edge_list_path <- "edge_list.csv"
write_csv(edges, edge_list_path)

# Read it
edges_in <- read_csv(edge_list_path, show_col_types = FALSE)
g_from_file <- graph_from_data_frame(edges_in, directed = FALSE)

g_from_file


## Next steps (we'll do after)

Your original `Lesson01_A` calls helper functions from `Functions.py`. Once you're ready, we can:

- Port the **exact helpers** to R (or rewrite them idiomatically)
- Replace the toy networks here with the **same datasets** used in the course
- Match every exercise cell-by-cell against the Python notebook
